# Interactive Spark SQL Queries
Run the cell below to initialize the Spark Session and load the `customers`, `orders`, and `products` tables into memory. Then, you can easily test single queries in the cells below!

In [1]:
import os
import sys

# Ensure the notebook can find PySpark in our virtual environment FIRST
venv_site_packages = os.path.join(os.getcwd(), 'venv', 'Lib', 'site-packages')
if venv_site_packages not in sys.path:
    sys.path.insert(0, venv_site_packages)

# Set HADOOP_HOME and JAVA_HOME
current_dir = os.getcwd()
os.environ['HADOOP_HOME'] = os.path.join(current_dir, 'hadoop')
os.environ['JAVA_HOME'] = os.path.join(current_dir, 'jdk-17')

from pyspark.sql import SparkSession

# Initialize Spark
spark = SparkSession.builder \
    .appName("InteractiveSQL") \
    .master("local[*]") \
    .getOrCreate()

# Load Datasets
customers_df = spark.read.csv("Sep_1/customers_dataset.csv", header=True, inferSchema=True)
orders_df = spark.read.csv("Sep_1/orders_dataset.csv", header=True, inferSchema=True)
products_df = spark.read.csv("Sep_1/products_dataset.csv", header=True, inferSchema=True)

# Create Temp Views
customers_df.createOrReplaceTempView("customers")
orders_df.createOrReplaceTempView("orders")
products_df.createOrReplaceTempView("products")

print("\u2705 Spark is Ready! Temp Tables available: 'customers', 'orders', 'products'")

✅ Spark is Ready! Temp Tables available: 'customers', 'orders', 'products'


### Test Your Queries Below

In [2]:
query = """
SELECT * 
FROM orders
LIMIT 5
"""

# Run and show results
spark.sql(query).show()

+--------+-----------+----------+----------+----------+--------+----------+-------------------+------------+------------+------------------+
|order_id|customer_id|product_id|order_date| ship_date|quantity|unit_price|discount_percentage|payment_mode|order_status|      sales_amount|
+--------+-----------+----------+----------+----------+--------+----------+-------------------+------------+------------+------------------+
|       1|         45|        27|2025-06-16|2025-06-22|       4|      6479|                 15|  Debit Card|   Delivered|           22028.6|
|       2|         64|        19|2026-02-14|2026-02-21|       3|     27477|                 22|         UPI|     Shipped|          64296.18|
|       3|         15|        50|2026-03-23|2026-03-30|       3|     17041|                 10|         UPI|     Shipped|46010.700000000004|
|       4|         47|        40|2025-10-08|2025-10-16|       1|     12793|                 16| Net Banking|   Delivered|10746.119999999999|
|       5|   

In [3]:
query = """
SELECT c.customer_id, c.customer_name, ROUND(SUM(o.sales_amount), 2) as total_revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_revenue DESC
LIMIT 10
"""

# Run and show results
spark.sql(query).show()


+-----------+----------------+-------------+
|customer_id|   customer_name|total_revenue|
+-----------+----------------+-------------+
|         93|Shirley Harrison|    845556.64|
|          6|Theresa Ferguson|    720304.25|
|         11|   Darrell Jones|    592001.09|
|         45|    Jason Nelson|    591651.49|
|         80|    Jason Garcia|    588679.86|
|        106|    Shane Miller|     581160.2|
|         52| Jennifer Hunter|    565407.64|
|         56|   Matthew Evans|    532683.28|
|         64|     Nicole Meza|    503299.37|
|        104| Alison Mcdonald|    497268.72|
+-----------+----------------+-------------+



In [17]:
query = """
SELECT DATE_FORMAT(order_date, 'yyyy-MM') as month, ROUND(SUM(sales_amount), 2) as total_revenue
FROM orders
GROUP BY DATE_FORMAT(order_date, 'yyyy-MM')
ORDER BY month ASC
"""

# Run and show results
spark.sql(query).show()


+-------+-------------+
|  month|total_revenue|
+-------+-------------+
|2025-05|    2780923.5|
|2025-06|   3047238.73|
|2025-07|   2519717.01|
|2025-08|   3116055.63|
|2025-09|   2073947.14|
|2025-10|   2432460.02|
|2025-11|   3146286.35|
|2025-12|   2698370.84|
|2026-01|   2728039.23|
|2026-02|   1875279.53|
|2026-03|   3167027.82|
|2026-04|   2734291.43|
|2026-05|    738030.75|
+-------+-------------+



In [5]:
query = """
WITH DailySales AS (
    SELECT order_date, ROUND(SUM(sales_amount), 2) as daily_revenue
    FROM orders
    GROUP BY order_date
)
SELECT 
    order_date, 
    daily_revenue, 
    ROUND(SUM(daily_revenue) OVER (ORDER BY order_date ASC), 2) as cumulative_revenue
FROM DailySales
ORDER BY order_date ASC
"""

# Run and show results
spark.sql(query).show()


+----------+-------------+------------------+
|order_date|daily_revenue|cumulative_revenue|
+----------+-------------+------------------+
|2025-05-07|    179179.04|         179179.04|
|2025-05-08|     81359.38|         260538.42|
|2025-05-09|     75555.24|         336093.66|
|2025-05-10|    490804.75|         826898.41|
|2025-05-12|    329271.19|         1156169.6|
|2025-05-14|     70558.95|        1226728.55|
|2025-05-16|     10617.62|        1237346.17|
|2025-05-17|     59956.93|         1297303.1|
|2025-05-19|    254054.68|        1551357.78|
|2025-05-21|     76966.06|        1628323.84|
|2025-05-22|    162546.86|         1790870.7|
|2025-05-23|     54300.09|        1845170.79|
|2025-05-24|    465520.33|        2310691.12|
|2025-05-25|     262350.4|        2573041.52|
|2025-05-26|     114203.0|        2687244.52|
|2025-05-27|     15765.52|        2703010.04|
|2025-05-30|     49703.46|         2752713.5|
|2025-05-31|      28210.0|         2780923.5|
|2025-06-01|     83541.23|        

In [6]:
query = """
SELECT 
    customer_id, 
    order_id, 
    order_date, 
    sales_amount AS current_purchase_amount,
    LAG(sales_amount) OVER (PARTITION BY customer_id ORDER BY order_date ASC) AS previous_purchase_amount,
    ROUND(sales_amount - LAG(sales_amount) OVER (PARTITION BY customer_id ORDER BY order_date ASC), 2) AS difference_from_previous
FROM orders
ORDER BY customer_id, order_date ASC
"""

# Run and show results
spark.sql(query).show()


+-----------+--------+----------+-----------------------+------------------------+------------------------+
|customer_id|order_id|order_date|current_purchase_amount|previous_purchase_amount|difference_from_previous|
+-----------+--------+----------+-----------------------+------------------------+------------------------+
|          1|     251|2025-09-19|               126064.4|                    NULL|                    NULL|
|          1|     424|2025-10-08|               30719.96|                126064.4|               -95344.44|
|          2|     443|2025-10-06|              108951.48|                    NULL|                    NULL|
|          2|     306|2025-11-21|               31602.42|               108951.48|               -77349.06|
|          2|     174|2025-12-04|     196283.40000000002|                31602.42|               164680.98|
|          2|     378|2026-03-01|                59844.0|      196283.40000000002|               -136439.4|
|          3|     402|2025-0

In [7]:
query = """
SELECT 
    customer_id, 
    order_id, 
    order_date, 
    sales_amount AS current_purchase_amount,
    LEAD(sales_amount) OVER (PARTITION BY customer_id ORDER BY order_date ASC) AS next_purchase_amount,
    LEAD(order_date) OVER (PARTITION BY customer_id ORDER BY order_date ASC) AS next_purchase_date
FROM orders
ORDER BY customer_id, order_date ASC
"""

# Run and show results
spark.sql(query).show()


+-----------+--------+----------+-----------------------+--------------------+------------------+
|customer_id|order_id|order_date|current_purchase_amount|next_purchase_amount|next_purchase_date|
+-----------+--------+----------+-----------------------+--------------------+------------------+
|          1|     251|2025-09-19|               126064.4|            30719.96|        2025-10-08|
|          1|     424|2025-10-08|               30719.96|                NULL|              NULL|
|          2|     443|2025-10-06|              108951.48|            31602.42|        2025-11-21|
|          2|     306|2025-11-21|               31602.42|  196283.40000000002|        2025-12-04|
|          2|     174|2025-12-04|     196283.40000000002|             59844.0|        2026-03-01|
|          2|     378|2026-03-01|                59844.0|                NULL|              NULL|
|          3|     402|2025-06-03|               32065.68|  2446.9900000000002|        2025-09-18|
|          3|     34

In [8]:
query = """
WITH CustomerMonths AS (
    SELECT DISTINCT 
        customer_id, 
        DATE_TRUNC('month', order_date) AS order_month
    FROM orders
),
NextOrderMonths AS (
    SELECT 
        customer_id, 
        order_month,
        LEAD(order_month) OVER (PARTITION BY customer_id ORDER BY order_month) AS next_order_month
    FROM CustomerMonths
)
SELECT DISTINCT 
    c.customer_id, 
    c.customer_name
FROM NextOrderMonths n
JOIN customers c ON n.customer_id = c.customer_id
WHERE MONTHS_BETWEEN(n.next_order_month, n.order_month) = 1
ORDER BY c.customer_id
"""

# Run and show results
spark.sql(query).show()


+-----------+-------------------+
|customer_id|      customer_name|
+-----------+-------------------+
|          1|    Zachary Randall|
|          2|      Morgan Wilson|
|          6|   Theresa Ferguson|
|          8|      Michael Sloan|
|         11|      Darrell Jones|
|         13|     William Bailey|
|         14|     Lisa Lopez DDS|
|         15|     Emily Galloway|
|         16|         Ryan Cohen|
|         18|      Jaime Vasquez|
|         19|      Hector Gordon|
|         20|       Paul Stevens|
|         24|         Karla Diaz|
|         25|           Jesse Le|
|         29|        Teresa Mann|
|         31|        Mario Flynn|
|         32|     Robert Jackson|
|         33|Mrs. Ashley Andrews|
|         34|  Marissa Mcfarland|
|         38|        Sarah Klein|
+-----------+-------------------+
only showing top 20 rows



In [9]:
query = """
WITH ProductSales AS (
    SELECT 
        p.category,
        p.product_id,
        p.product_name,
        SUM(o.sales_amount) as total_sales
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category, p.product_id, p.product_name
),
RankedProducts AS (
    SELECT 
        category,
        product_id,
        product_name,
        total_sales,
        ROW_NUMBER() OVER (PARTITION BY category ORDER BY total_sales DESC) as rank
    FROM ProductSales
)
SELECT 
    category,
    product_id,
    product_name,
    ROUND(total_sales, 2) as total_sales
FROM RankedProducts
WHERE rank = 1
ORDER BY category
"""

# Run and show results
spark.sql(query).show()


+-----------+----------+--------------+-----------+
|   category|product_id|  product_name|total_sales|
+-----------+----------+--------------+-----------+
|Electronics|        44|Worker Product|  742170.85|
|    Fashion|        43| Build Product|  1215711.4|
|  Furniture|         7| Place Product| 1193462.47|
|    Grocery|        19|  Fill Product|  763266.63|
|     Sports|        27| Visit Product|  749711.32|
+-----------+----------+--------------+-----------+



In [10]:
query = """
SELECT 
    ROUND(AVG(DATEDIFF(ship_date, order_date)), 2) AS average_shipping_delay_days
FROM orders
WHERE ship_date IS NOT NULL
"""

# Run and show results
spark.sql(query).show()


+---------------------------+
|average_shipping_delay_days|
+---------------------------+
|                       5.67|
+---------------------------+



In [11]:
query = """
SELECT 
    c.customer_id, 
    c.customer_name, 
    c.city, 
    c.state
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL
"""

# Run and show results
spark.sql(query).show()


+-----------+-------------+----+----------+
|customer_id|customer_name|city|     state|
+-----------+-------------+----+----------+
|         37|   Robin Cobb|Pune|Tamil Nadu|
+-----------+-------------+----+----------+



In [12]:
query = """
WITH CustomerRevenue AS (
    SELECT 
        c.customer_id, 
        c.customer_name, 
        SUM(o.sales_amount) AS total_revenue
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.customer_name
)
SELECT 
    RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
    customer_id, 
    customer_name, 
    ROUND(total_revenue, 2) AS total_revenue
FROM CustomerRevenue
ORDER BY revenue_rank
"""

# Run and show results
spark.sql(query).show()


+------------+-----------+-----------------+-------------+
|revenue_rank|customer_id|    customer_name|total_revenue|
+------------+-----------+-----------------+-------------+
|           1|         93| Shirley Harrison|    845556.64|
|           2|          6| Theresa Ferguson|    720304.25|
|           3|         11|    Darrell Jones|    592001.09|
|           4|         45|     Jason Nelson|    591651.49|
|           5|         80|     Jason Garcia|    588679.86|
|           6|        106|     Shane Miller|     581160.2|
|           7|         52|  Jennifer Hunter|    565407.64|
|           8|         56|    Matthew Evans|    532683.28|
|           9|         64|      Nicole Meza|    503299.37|
|          10|        104|  Alison Mcdonald|    497268.72|
|          11|         78|Joshua Washington|    494396.52|
|          12|         48| Kristine Schmidt|    493638.62|
|          13|         94|     Sheila Stone|    493296.89|
|          14|        114|    Alan Anderson|    489207.5

In [13]:
query = """
WITH CustomerPurchases AS (
    SELECT 
        customer_id, 
        order_id, 
        order_date, 
        sales_amount AS current_sales,
        LAG(sales_amount) OVER (PARTITION BY customer_id ORDER BY order_date ASC) AS previous_sales
    FROM orders
)
SELECT 
    c.customer_name,
    p.customer_id, 
    p.order_id, 
    p.order_date, 
    ROUND(p.current_sales, 2) AS current_sales, 
    ROUND(p.previous_sales, 2) AS previous_sales
FROM CustomerPurchases p
JOIN customers c ON p.customer_id = c.customer_id
WHERE p.current_sales < p.previous_sales
ORDER BY p.customer_id, p.order_date ASC
"""

# Run and show results
spark.sql(query).show()


+----------------+-----------+--------+----------+-------------+--------------+
|   customer_name|customer_id|order_id|order_date|current_sales|previous_sales|
+----------------+-----------+--------+----------+-------------+--------------+
| Zachary Randall|          1|     424|2025-10-08|     30719.96|      126064.4|
|   Morgan Wilson|          2|     306|2025-11-21|     31602.42|     108951.48|
|   Morgan Wilson|          2|     378|2026-03-01|      59844.0|      196283.4|
|      Troy Brown|          3|     347|2025-09-18|      2446.99|      32065.68|
|Theresa Ferguson|          6|      35|2025-06-09|     31280.64|      80970.72|
|Theresa Ferguson|          6|     190|2025-07-05|     12734.26|      112581.0|
|Theresa Ferguson|          6|     417|2025-08-31|     23609.58|      72337.16|
|Theresa Ferguson|          6|     108|2026-01-24|      9771.84|      145645.2|
|  Yvonne Carroll|          7|     100|2025-12-11|     39257.84|      75241.65|
|   Michael Sloan|          8|     166|2

In [15]:
query = """
SELECT 
    DATE_FORMAT(order_date, 'EEEE') AS weekday, 
    ROUND(SUM(sales_amount), 2) AS total_revenue
FROM orders
GROUP BY DATE_FORMAT(order_date, 'EEEE')
ORDER BY total_revenue DESC
"""

# Run and show results
spark.sql(query).show()


+---------+-------------+
|  weekday|total_revenue|
+---------+-------------+
|   Monday|    5655873.8|
| Saturday|   5031840.67|
|Wednesday|   4647350.79|
|  Tuesday|   4534374.49|
|   Sunday|   4483950.16|
| Thursday|   4423944.61|
|   Friday|   4280333.46|
+---------+-------------+

